# Disease relevance and mechanism classes of the translated lincRNA ORFs

Continues [Liver_lincRNA_ORFs.ipynb](Liver_lincRNA_ORFs.ipynb) from the lincorfz run `v2`: every ORF of the run is
annotated with quality flags, disease-relevance measures D1-D6 (summarized as an HCC score and a chronic liver
disease score) and mechanism classes M1-M6 (encoded peptide, translation-coupled RNA regulation, cis regulation,
nuclear trans function, cytoplasmic trans function, small-RNA host). The definitions are in
[lincdisease/LOGIC.md](lincdisease/LOGIC.md), the Methods in `lincdisease_Methods.docx` of the output directory.

* **Heavy computations** are done by the Snakemake workflow [lincdisease/workflow](lincdisease/workflow/README.md),
  driven by `lincdisease/run.sh`: downloads on the login node, everything else as Slurm jobs.
* **This notebook** prints the commands, reads the result tables and plots them. It can itself be executed as a
  Slurm job (last section).
* **Paths** come from `lincdisease/config/config.yaml`, relative to the root directory of the workflow
  (environment variable `LINCDISEASE_ROOT`; default: `$WF_RUNS_DIR/Liver_lincRNA_ORFs_Claude`, the lincorfz output
  directory `OUT_BASE` of `Liver_lincRNA_ORFs.ipynb`).
* **Outputs**: this notebook is kept in the repository without outputs. The executed copy, with all tables and
  figures, is `notebook/Liver_lincRNA_ORFs_disease.html` in the output directory (section 4 makes it); the figures
  are also saved as PDF in `figures/` there.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy import stats
os.environ.setdefault("MPLCONFIGDIR", tempfile.mkdtemp())   # the home directory may be read-only
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

LDIR = Path(os.environ.get("LINCDISEASE_DIR", "lincdisease")).resolve()      # lincdisease/ of the repository
ROOT = os.environ.get("LINCDISEASE_ROOT") or os.environ.get("WF_RUNS_DIR") and f"{os.environ['WF_RUNS_DIR']}/Liver_lincRNA_ORFs_Claude"
assert ROOT, "set LINCDISEASE_ROOT (the lincorfz output directory) or WF_RUNS_DIR"
ROOT = Path(ROOT).resolve()
sys.path.insert(0, str(LDIR))
from lincdisease import chronic                                               # noqa: E402  (Govaere annotation)

cfg = yaml.safe_load(open(LDIR / "config" / "config.yaml"))
at_root = lambda p: Path(p) if os.path.isabs(p) else ROOT / p
OUT = at_root(cfg["out_dir"])
GOV = at_root(cfg["govaere"]["dir"])
PY = at_root(cfg["python"])
RUN_SH = LDIR / "run.sh"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = {"M1": "encoded peptide", "M2": "translation-coupled RNA", "M3": "cis regulation",
               "M4": "nuclear trans", "M5": "cytoplasmic trans", "M6": "small-RNA host"}
print("root:", ROOT, "\noutputs:", OUT)

## 1. Conda environments and inputs

Compute nodes have no internet access, so the environments are built once on a login node from
`lincdisease/envs/` (conda-forge and bioconda channels directly):

* `lincdisease`: the package, its tests and this notebook (`ipykernel`, `nbclient`, `nbconvert`)
* `lincdisease_peptide`: MHCflurry and the sequence-feature tools of the `peptide` step
* `snakemake`: Snakemake 9 with the cluster-generic executor

The workflow starts from the finished lincorfz run and reads its results in place (`runs/v2/results` under the root; see
the workflow README).

In [ ]:
for env in ["lincdisease", "lincdisease_peptide"]:
    print(f"mamba env create -p {ROOT}/conda_envs/{env} -f {LDIR}/envs/{env}.yml")
print(f"mamba create -p {ROOT}/conda_envs/snakemake -c conda-forge -c bioconda python=3.12 snakemake-minimal=9.27.1 "
      "snakemake-executor-plugin-cluster-generic=1.0.9")

inputs = {k: at_root(cfg[k]) for k in ["results_dir", "candidates", "peptides_dir", "phylop"]}
display(pd.DataFrame({"path": {k: str(p) for k, p in inputs.items()},
                      "exists": {k: p.exists() for k, p in inputs.items()}}))

## 2. Run the workflow

From the root directory on a login node. `start` runs in the background (the downloads need the network; all
other steps are Slurm jobs) and resumes where an earlier run stopped; a step reruns when its inputs or the
code it uses changes (docstrings and comments excepted; see the workflow README).

In [ ]:
print(f"cd {ROOT}")
for cmd in ["plan", "start", "status"]:
    print(f"{RUN_SH} {cmd}")
print(f"{RUN_SH} redo mechanism     # rerun a step and everything downstream of it")

## 3. Results

### 3.1 Quality layer: confidence tiers and copy attribution

Every ORF gets flags (segmental duplication, GRCh38 false duplication, homology to annotated proteins, antisense
position, small-RNA overlap, multimapper dominance, coding-gene extension), a copy-attribution confidence for ORFs
with paralogous copies, and a tier: clean / caution / expressed_paralog / likely_artefact.

In [ ]:
qc = pd.read_csv(OUT / "qc.orfs.tsv", sep="\t")
tiers = ["clean", "caution", "expressed_paralog", "likely_artefact"]
t = pd.crosstab(qc["confidence_tier"], qc["is_candidate"].map({True: "candidates", False: "other ORFs"}))
display(t.reindex(tiers).fillna(0).astype(int))

fig, ax = plt.subplots(figsize=(5, 3.2))
for tier, color in zip(tiers, ["#2a7f62", "#d9a441", "#c0504d", "#7f7f7f"]):
    v = qc.loc[qc["confidence_tier"] == tier, "copy_attribution"].dropna()
    ax.hist(v, bins=np.linspace(0, 1, 21), histtype="step", lw=1.5, color=color, label=f"{tier} ({len(v)})")
ax.set(xlabel="copy-attribution confidence", ylabel="ORFs", yscale="log")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "qc_copy_attribution.pdf", bbox_inches="tight")

### 3.2 Disease relevance: D1-D6, HCC and chronic liver disease scores

D1 differential expression (HCC: TCGA-LIHC tumour vs adjacent normal of the same patient; chronic: fibrosis stage
in MASLD cohorts and HCV status), D2 survival, D3 somatic copy number, D4 liver-trait GWAS and eQTLs, D5 literature,
D6 liver context. Expression is scaled by gene-based size factors (hepatocyte transcripts, which dominate liver RNA,
lose their share in tumours). Components are percentiles among all quantified loci (0.5 = typical lincRNA) of the
**magnitude** of the signed statistics: a locus strongly down in tumours scores as high as one strongly up, and
the direction is in `hcc_direction` / `chronic_direction`. D1 HCC, D1 chronic and D2 are first centred on their
background median, as most loci shift a little in the same direction (up in tumours, down in chronic disease, higher
with worse survival); the `stat_` columns are uncentred. D1 chronic (`stat_D1_chronic`) is a Stouffer z over the
Hoang, Govaere and HCV cohorts, not a correlation. D1 HCC is replicated (reported, not scored) in four recount3
cohorts of tumours and adjacent liver of the same patients (GSE77314, GSE77509, GSE94660, GSE56545; 104 pairs):
`hcc_replicated` = replication FDR < 0.05 and the replication r and D1 HCC on the same side of their background
medians.

The heatmap shows the components entering the scores (percentiles: 0.5 = a typical locus, 1 = the most extreme;
they have no direction). Below it, the candidates are compared with the background: if the candidates' replication
rate is not significantly higher than the background's (Fisher test), replication does not by itself single them
out, and the Mann-Whitney tests say whether the candidates score above a typical lincRNA locus.

In [ ]:
orfs = pd.read_csv(OUT / "lincdisease.orfs.tsv", sep="\t", low_memory=False)
cand = pd.read_csv(OUT / "lincdisease.candidates.tsv", sep="\t")
comps = [f"comp_D{i}" for i in range(1, 7)]
c = cand.sort_values("hcc_score", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 9), gridspec_kw={"width_ratios": [3, 1.3]})
im = axes[0].imshow(c[comps].to_numpy(float), aspect="auto", cmap="viridis", vmin=0, vmax=1)
axes[0].set_xticks(range(6), [s.replace("comp_", "") for s in comps])
axes[0].set_yticks(range(len(c)), c["orf_label"], fontsize=7)
fig.colorbar(im, ax=axes[0], shrink=0.4, label="component (percentile)")
y = np.arange(len(c))
axes[1].barh(y - 0.2, c["hcc_score"], height=0.4, color="#c0504d", label="HCC")
axes[1].barh(y + 0.2, c["chronic_score"], height=0.4, color="#4f81bd", label="chronic")
axes[1].axvline(0.5, color="grey", ls=":")
axes[1].set(yticks=[], ylim=axes[0].get_ylim(), xlabel="score", xlim=(0, 1))
axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "candidates_disease_components.pdf", bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
loci = orfs.drop_duplicates("locus_id")
ax.scatter(loci["hcc_score"], loci["chronic_score"], s=3, color="lightgrey", label="all loci")
cl = cand.drop_duplicates("locus_id")
ax.scatter(cl["hcc_score"], cl["chronic_score"], s=18, color="#c0504d", label="candidate loci")
for _, r in cl.nlargest(8, "hcc_score").iterrows():
    ax.annotate(r["locus_name"], (r["hcc_score"], r["chronic_score"]), fontsize=7)
for _, r in cl.nlargest(5, "chronic_score").iterrows():
    ax.annotate(r["locus_name"], (r["hcc_score"], r["chronic_score"]), fontsize=7)
ax.set(xlabel="HCC score", ylabel="chronic liver disease score")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "hcc_vs_chronic_score.pdf", bbox_inches="tight")
cols = ["orf_label", "confidence_tier", "hcc_score", "chronic_score", "log2FC_paired", "fdr_paired",
        "hcc_replicated", "stat_D1_chronic", "chronic_fdr", "HR_OS", "fdr_OS", "gwas_top_trait", "n_liver"]
dis = pd.read_csv(OUT / "disease.loci.tsv", sep="\t", index_col=0, low_memory=False)
bg = dis[~dis["is_control"] & ~dis["is_candidate"]]
rep_c = cl["hcc_replicated"].dropna().astype(bool)
rep_b = bg["hcc_replicated"].dropna().astype(bool)
odds, p = stats.fisher_exact([[rep_c.sum(), (~rep_c).sum()], [rep_b.sum(), (~rep_b).sum()]])
print(f"HCC replication: {rep_c.sum()} of {len(rep_c)} tested candidate loci ({rep_c.mean():.0%}) vs "
      f"{rep_b.mean():.0%} of the background loci (odds ratio {odds:.1f}, Fisher p = {p:.2f})")
for s in ["hcc_score", "chronic_score", "disease_score"]:
    print(f"{s}: candidates median {cl[s].median():.2f}, background {bg[s].median():.2f}, Mann-Whitney p = "
          f"{stats.mannwhitneyu(cl[s].dropna(), bg[s].dropna()).pvalue:.1e}")
# stat_D1_chronic is a Stouffer z (uncentred), not a correlation
display(cand[cols].sort_values("hcc_score", ascending=False).round(3))

### 3.3 Positive and negative controls

One or two known genes per measure and score, with the expectation fixed before scoring. Mechanism controls that
are not lincorfz loci (MTLN, SPAAR) cannot be scored. The mechanism controls are feature checks: they show that the
features of a class are defined as intended (several pass on annotation features alone, such as divergent position
or small-RNA host status, and M4 on the very high abundance of MALAT1 and NEAT1), not that the mechanism is
detected. The thresholds of every check are defined once in `controls.PANEL`; a check whose values are missing
is "not scored". Failures are reported as they are, not fixed by tuning: read the `result` and `value` columns
below for which controls fail and why (e.g. a control that is not up in tumours in these data).

In [ ]:
ctrl = pd.read_csv(OUT / "controls.tsv", sep="\t")
status = ctrl["pass"].map({True: "pass", False: "FAIL"}).fillna("not scored")
display(ctrl.assign(result=status).drop(columns="pass").style.apply(
    lambda r: ["background-color: #f4cccc" if r["result"] == "FAIL" else ""] * len(r), axis=1))
print(status.value_counts().to_dict())

### 3.4 Mechanism classes

Each class score is the mean of its feature percentiles (missing features shrunk to 0.5). Classes are compared on a
rank-based standardized score `M*_z`: the normal quantile of the class score's percentile among the ORFs with data
for the class (locus-weighted). The dominant class is the class with the highest z if z > 0 and it exceeds the
runner-up by 0.5, otherwise `none`. Features that rise with ORF length are ranked within ORF-length bins, and
features that follow HepG2 expression within HepG2-expression bins, so the classes do not reward length or
expression as such; M1 features that a paralogous copy could supply are weighted by copy attribution. M2 uses the
NMD substrate rule and the measured NMD sensitivity (3.5).

The last table is a diagnostic: the Spearman correlation over all ORFs of each class z with HepG2 expression and
with ORF length. Values near 0 mean the class is not a proxy for either. M4 is expected to follow HepG2 expression,
as abundance is its defining feature; a large correlation for another class would mean that class mostly sorts
ORFs by expression or length.

In [ ]:
mech = pd.read_csv(OUT / "mechanism.orfs.tsv", sep="\t", low_memory=False)
classes = list(CLASS_NAMES)
d = pd.concat({"all ORFs": mech["dominant_class"].value_counts(),
               "candidates": mech.loc[mech["is_candidate"], "dominant_class"].value_counts()}, axis=1)
display(d.reindex(classes + ["none"]).fillna(0).astype(int))

mc = mech.loc[mech["is_candidate"]].set_index("orf_label")
mc = mc.sort_values(["dominant_class"] + [f"{k}_z" for k in classes])
z = mc[[f"{k}_z" for k in classes]].to_numpy(float)
fig, ax = plt.subplots(figsize=(7, 10))
im = ax.imshow(z, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(6), [f"{k}\n{CLASS_NAMES[k]}" for k in classes], fontsize=7)
ax.set_yticks(range(len(mc)), [f"{o}  [{d}]" for o, d in zip(mc.index, mc["dominant_class"])], fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.4, label="standardized class score (z)")
fig.savefig(FIG / "candidates_mechanism_z.pdf", bbox_inches="tight")

# raw_m4_abundance is log2 TPM in HepG2
diag = pd.DataFrame({f"{k}_z": {
    "HepG2 log2 TPM": stats.spearmanr(mech[f"{k}_z"], mech["raw_m4_abundance"], nan_policy="omit")[0],
    "ORF length": stats.spearmanr(mech[f"{k}_z"], mech["raw_orf_length"], nan_policy="omit")[0]} for k in classes})
display(diag.round(2))

### 3.5 Measured NMD sensitivity (M2)

HeLa UPF1 knock-down and SMG6 + SMG7 double knock-down vs scrambled controls (Colombo et al. 2017, recount3
SRP083135); per locus the mean log2 fold change of the two knock-downs, shrunk toward 0 by its standard error
(`nmd_log2fc`; the raw value is `nmd_log2fc_raw`). Loci whose ORF places the stop codon > 50 nt upstream of the last
exon junction (predicted NMD substrates) should rise upon NMD inhibition. A locus counts as a predicted substrate
if any of its ORFs is one. If the predicted substrates rise clearly more than the other loci (left panel), and the
known NMD targets among the controls lie up in both knock-downs (right panel, raw per-condition fold changes), the
measurement behaves as expected and can be used as an M2 feature.

In [ ]:
nmd = pd.read_csv(OUT / "nmd.loci.tsv", sep="\t").set_index("feature_id")
pred = mech.groupby("locus_id")["raw_m2_nmd"].max()       # a locus is a predicted substrate if any ORF is
v = nmd.join(pred.rename("predicted_nmd"), how="inner").dropna(subset=["nmd_log2fc", "predicted_nmd"])
a, b = v.loc[v["predicted_nmd"] == 1, "nmd_log2fc"], v.loc[v["predicted_nmd"] == 0, "nmd_log2fc"]
print(f"predicted NMD substrates: n={len(a)}, median log2FC {a.median():.2f}; others: n={len(b)}, "
      f"median {b.median():.2f}; Mann-Whitney p = {stats.mannwhitneyu(a, b).pvalue:.1e}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
bins = np.linspace(-4, 4, 41)
axes[0].hist(b.clip(-4, 4), bins=bins, density=True, histtype="step", lw=1.5, color="grey", label="other loci")
axes[0].hist(a.clip(-4, 4), bins=bins, density=True, histtype="step", lw=1.5, color="#c0504d",
             label="predicted NMD substrate")
axes[0].set(xlabel="shrunken log2FC upon NMD inhibition (mean of UPF1 KD, SMG6+7 dKD)", ylabel="density")
axes[0].legend(frameon=False, fontsize=8)
axes[1].scatter(nmd["log2fc_UPF1_kd"], nmd["log2fc_SMG6_SMG7_kd"], s=3, color="lightgrey")
for fid, r in nmd.loc[nmd.index.str.startswith("ctrl:")].iterrows():
    axes[1].scatter(r["log2fc_UPF1_kd"], r["log2fc_SMG6_SMG7_kd"], s=12, color="k")
    axes[1].annotate(fid[5:], (r["log2fc_UPF1_kd"], r["log2fc_SMG6_SMG7_kd"]), fontsize=7)
axes[1].set(xlabel="log2FC UPF1 KD", ylabel="log2FC SMG6 + SMG7 dKD", title="control genes labelled")
fig.tight_layout()
fig.savefig(FIG / "nmd_sensitivity_validation.pdf", bbox_inches="tight")

### 3.6 ORFs supported by published CRISPR screens

ORFs with a screen value >= 1: viability phenotype of an ORF-targeting guide in Prensner et al. 2021 (2 if the
phenotype is ORF-specific) or a hit of Chen et al. 2020 (re-called against the non-targeting null).

In [ ]:
scr = pd.read_csv(OUT / "lincdisease.screen_supported.tsv", sep="\t")
print(f"{len(scr)} ORFs, {scr['is_candidate'].sum()} of them candidates")
cols = ["orf_label", "is_candidate", "confidence_tier", "crispr_screen_value", "prensner_hepg2_lfc",
        "chen_K562_phenotype", "chen_iPSC_phenotype", "phylop_orf_minus_nonorf", "hcc_score", "chronic_score",
        "dominant_class"]
display(scr[cols].sort_values(["crispr_screen_value", "hcc_score"], ascending=False).round(3))

### 3.7 Govaere et al. 2020 NAFLD cohort (second fibrosis cohort of D1 chronic)

216 liver biopsies (GSE135251, PRJNA558102; NAFL and NASH with fibrosis stage F0-F4, plus controls), not in
recount3: aligned with STAR to GRCh38 / GENCODE v42 by the `govaere` target of the workflow and archived as CRAM 3.1
(`<govaere dir>/cram/`); 215 are used (`govaere.exclude` in the config: the read 2 file of one NASH F2 run is
corrupt at ENA). The workflow counts the loci in each CRAM over the same regions as in the other cohorts (rule
`govaere_locus_cov`), and the `chronic` step tests them against fibrosis stage with the D1 chronic statistic, next
to Hoang et al. 2019 (same design; in the Stouffer z each cohort is weighted by the square root of its samples).

Below, first the samples per histological group (the `group in paper` column is the group given in the GEO sample
characteristics) with how many were aligned, quantified and used; all groups should be close to complete. Then the
agreement of the two fibrosis cohorts: a clearly positive Spearman correlation of their per-feature statistics
means the fibrosis association replicates between independent cohorts, and COL1A1 should be positive in both.

In [ ]:
# commands (login node): sample annotation from GEO (once), then the workflow aligns, counts and reruns D1 chronic
print(f"curl -sS -f -o {GOV}/metadata/GSE135251_samples.soft.txt "
      "'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE135251&targ=gsm&form=text&view=brief'")
print(f"{RUN_SH} start")

# the ENA run table is part of the repository: config govaere.runs is relative to lincdisease/, not to the root
gov = chronic.govaere_samples(LDIR / cfg["govaere"]["runs"], GOV / "metadata" / "GSE135251_samples.soft.txt")
gov["aligned"] = [(GOV / "cram" / f"{r}.ok").exists() for r in gov.index]
gov["quantified"] = [(GOV / "locus_cov" / f"{r}.tsv").exists() for r in gov.index]
gov["used"] = gov["quantified"] & ~gov.index.isin(cfg["govaere"].get("exclude") or [])
display(gov.groupby("group in paper")[["aligned", "quantified", "used"]].agg(["sum", "size"]))

In [ ]:
# Govaere vs Hoang et al. 2019: fibrosis statistics of D1 chronic
d1c = pd.read_csv(OUT / "D1.chronic.tsv", sep="\t").set_index("feature_id")
both = d1c.dropna(subset=["govaere_r", "hoang_r"])
rho = stats.spearmanr(both["govaere_r"], both["hoang_r"])
print(f"Govaere vs Hoang fibrosis statistics over {len(both)} features: Spearman {rho.statistic:.2f} "
      f"(p = {rho.pvalue:.1e})")
if "ctrl:COL1A1" in both.index:
    print("COL1A1:", both.loc["ctrl:COL1A1", ["govaere_r", "govaere_p", "hoang_r", "hoang_p"]].round(4).to_dict())
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(both["hoang_r"], both["govaere_r"], s=4, color="lightgrey")
cb = both.loc[both.index.isin(set(cand["locus_id"]))]
ax.scatter(cb["hoang_r"], cb["govaere_r"], s=18, color="#c0504d", label="candidate loci")
if "ctrl:COL1A1" in both.index:
    r = both.loc["ctrl:COL1A1"]
    ax.scatter(r["hoang_r"], r["govaere_r"], s=30, color="k", marker="^", label="COL1A1")
lim = (-0.8, 0.8)
ax.plot(lim, lim, ls=":", color="grey")
ax.set(xlim=lim, ylim=lim, xlabel="Hoang 2019: partial r with fibrosis stage",
       ylabel=f"Govaere 2020 ({int(d1c['govaere_n'].max())} runs): partial r with fibrosis stage",
       title=f"{len(both)} features, Spearman {rho.statistic:.2f} (p = {rho.pvalue:.1e})")
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG / "govaere_vs_hoang_fibrosis.pdf", bbox_inches="tight")

# D1_chronic: Stouffer z over the scored cohorts; chronic_r: their n-weighted mean partial r
cols = ["govaere_r", "govaere_p", "govaere_nas_rho", "hoang_r", "hoang_p", "D1_chronic", "chronic_r", "chronic_fdr"]
cg = cand.drop_duplicates("locus_id").set_index("locus_id")[["locus_name", "chronic_score"]].join(d1c[cols], how="left")
display(cg.sort_values("govaere_p").round(3))

### 3.8 Final table

`lincdisease.table.html` (output directory) holds the candidates and the screen-supported ORFs with all scores,
sortable, with column groups that can be hidden and an index of the scores; the tab-separated tables
`lincdisease.candidates.tsv`, `lincdisease.screen_supported.tsv` and `lincdisease.orfs.tsv` (all ORFs) have the
same columns. Candidates are ordered by the disease score without literature (`disease_score_noD5`), so
that well-studied loci are not favoured.

In [ ]:
print(OUT / "lincdisease.table.html")
cols = ["orf_label", "confidence_tier", "n_aa", "ribo_evidence", "disease_score_noD5", "hcc_score", "chronic_score",
        "actionability_score", "dominant_class"] + [f"{k}_z" for k in CLASS_NAMES]
display(cand[cols].round(2))        # in the order of lincdisease.candidates.tsv

## 4. Execute this notebook as a Slurm job

A copy is executed in the output directory, so that the notebook in the repository stays without outputs; the
executed copy is exported to HTML next to it.

In [ ]:
nb_dir = OUT / "notebook"
logs = nb_dir / "slurm_logs"
logs.mkdir(parents=True, exist_ok=True)
nb = "Liver_lincRNA_ORFs_disease.ipynb"
env = PY.parent.parent
repo = LDIR.parent                                      # Liver_lincRNA_ORFs/ of the repository
print(f"cp {repo / nb} {nb_dir}/ && \\\n"
      f"sbatch --time=01:00:00 --mem=16G --export=ALL,LINCDISEASE_ROOT={ROOT},LINCDISEASE_DIR={LDIR} --output={logs}/notebook_%j.out "
      f"--error={logs}/notebook_%j.err --wrap='bash {repo}/sbatch/execute_notebook.sbatch {nb_dir / nb} {env} && "
      f"{env}/bin/jupyter nbconvert --to html {nb_dir / nb}'")